# PECCAVI smoke test (Falcon `peccavi-baseline`), Kaggle 2x T4

5 PIE-Bench++ images, InstructPix2Pix, 4 sequential edits, tracks `none` + `peccavi`.
GPU 0 first checks one image end to end; if the unedited watermarked image is not detected (or the clean one is), everything stops.

**Settings (right panel):** Accelerator = GPU T4 x2, Internet = On, Add-ons > Secrets: `HF_TOKEN` attached (optional).
**Run:** Save Version > Save & Run All (Commit). Outputs: `/kaggle/working/peccavi_smoke/` (report in `report/`).
**Resume after a timeout/crash:** Add Input > Your Work > this notebook's previous version, then Save & Run All again; finished stages are skipped.

In [ ]:
FORK = "abhinavrdy4/Falcon"      # your fork
BRANCH = "peccavi-baseline"
REPO = "/tmp/Falcon"             # code lives outside /kaggle/working so it is not saved as output
OUT = "/kaggle/working/peccavi_smoke"
RUN_ARGS = ""                    # e.g. "--no-gate --set dataset.max_images=10" (see run_peccavi_smoke.py)
import os, subprocess, sys, time, json
T = {}
def sh(cmd, **kw):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, **kw).returncode

In [ ]:
# preflight: fail in seconds, not after setup, if the session has no internet or not 2 GPUs
import socket, shutil
try:
    socket.getaddrinfo("github.com", 443)
except OSError as e:
    raise RuntimeError(f"NO INTERNET ({e}). Kaggle gives internet (and GPUs) only to phone-verified accounts: "
                     "Settings > Phone Verification, then Session options > Internet: On.")
gpus = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines() \
    if shutil.which("nvidia-smi") else []
print("\n".join(gpus) or "no GPU")
if len(gpus) < 2:
    raise RuntimeError(f"need GPU T4 x2, found {len(gpus)} GPU(s): set Accelerator = GPU T4 x2 "
                     "(GPUs also need a phone-verified account)")

In [ ]:
t = time.time()
if os.path.exists(REPO):
    sh(f"git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}")
else:
    sh(f"git clone -q --depth 1 --branch {BRANCH} https://github.com/{FORK}.git {REPO}")
assert sh(f"git -C {REPO} log -1 --format='%H %s'") == 0, "clone failed"
T["clone_s"] = round(time.time() - t, 1)

In [ ]:
t = time.time()
assert sh('pip install -q "diffusers>=0.35" "transformers>=4.50" accelerate lpips open_clip_torch saliency shapely '
   'pytorch-msssim omegaconf tabulate wandb') == 0, "pip install failed"
T["install_s"] = round(time.time() - t, 1)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("no HF_TOKEN secret (fine: every model used is ungated):", type(e).__name__)
os.environ["WANDB_MODE"] = "disabled"
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")

In [ ]:
t = time.time()
assert sh(f"cd {REPO}/research && {sys.executable} scripts/kaggle/prefetch_models.py") == 0
T["prefetch_s"] = round(time.time() - t, 1)
os.makedirs(OUT, exist_ok=True)
json.dump(T, open(f"{OUT}/setup_times.json", "w"), indent=2)
print(T)

In [ ]:
# check (GPU 0) + both shards + report; per-stage logs in OUT/logs, wall times in OUT/run_times.json
rc = sh(f"cd {REPO}/research && {sys.executable} scripts/kaggle/run_peccavi_smoke.py --out {OUT} {RUN_ARGS}")
print("runner exit code", rc)

In [ ]:
from IPython.display import Markdown, Image, display
chk = f"{OUT}/gpu0/peccavi_smoke/peccavi_check.json"
if os.path.exists(chk):
    c = json.load(open(chk))
    print(json.dumps({k: c[k] for k in ("id", "quality", "image_info", "watermarked", "clean")}, indent=1)[:4000])
sh(f"tail -n 25 {OUT}/logs/gpu0_check.log")
rep = f"{OUT}/report/results.md"
if os.path.exists(rep):
    display(Markdown(open(rep).read()))
    display(Image(f"{OUT}/report/peccavi_fig1_retention_det_rate.png"))
else:
    sh(f"tail -n 40 {OUT}/logs/*.log")